In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import joblib

# 2. LOAD LOCATION DATA

df = pd.read_csv("valve_pressure_datanew.csv")
print("✅ Base Data loaded:", df.shape)  


# 3. SMART FEATURE ENGINEERING (The Efficiency Upgrade)

# Calculate the exact pressure drop between connected valves
leak_cols = [col for col in df.columns if col.startswith('Leak_')]

for leak in leak_cols:
    parts = leak.split('_') 
    if len(parts) == 3:
        valve1, valve2 = parts[1], parts[2]
        p1, p2 = f'Pressure_{valve1}', f'Pressure_{valve2}'
        
        # Calculate the absolute pressure drop between the two points
        if p1 in df.columns and p2 in df.columns:
            df[f'Drop_{valve1}_{valve2}'] = abs(df[p1] - df[p2])

print(f"⚡ Added {len(leak_cols)} new 'Pressure Drop' features.")


# 4. IDENTIFY TARGETS (Y) & FEATURES (X)

Y = df[leak_cols]
X = df.drop(columns=leak_cols + ["Active_Leaks"])


# 5. TRAIN-TEST SPLIT

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

# 6. TRAIN OPTIMIZED RANDOM FOREST

model = RandomForestClassifier(
    n_estimators=150,
    # 🔥 This is the magic parameter! It forces the trees to care 
    # significantly more about catching the '1's (the actual leaks).
    class_weight='balanced_subsample', 
    random_state=42,
    n_jobs=-1  # Uses all CPU cores
)

print("\n🚀 Training Optimized Random Forest Model...")
# Note: Random Forest handles multi-label natively, so no extra wrappers are needed!
model.fit(X_train, Y_train)
print("✅ Model trained successfully!")


# 7. EVALUATION

Y_pred = model.predict(X_test)

print("\n📊 New Classification Report (Watch the Recall scores!):\n")
print(classification_report(Y_test, Y_pred, target_names=leak_cols, zero_division=0))

# 8. SAVE MODEL

joblib.dump(model, "rf_optimized_leak_detector.pkl")
print("\n💾 Model saved as 'rf_optimized_leak_detector.pkl'")

✅ Base Data loaded: (10000, 51)
⚡ Added 16 new 'Pressure Drop' features.

🚀 Training Optimized Random Forest Model...
✅ Model trained successfully!

📊 New Classification Report (Watch the Recall scores!):

              precision    recall  f1-score   support

    Leak_A_B       1.00      0.70      0.83       205
    Leak_B_C       1.00      0.78      0.88       206
    Leak_B_D       0.97      0.66      0.79       192
    Leak_D_E       1.00      0.49      0.66       180
    Leak_E_H       1.00      0.63      0.77       164
    Leak_H_G       1.00      0.73      0.85       183
    Leak_D_F       1.00      0.81      0.90       180
    Leak_F_I       1.00      0.71      0.83       173
    Leak_F_J       1.00      0.73      0.85       200
    Leak_G_K       1.00      0.75      0.86       186
    Leak_H_L       1.00      0.82      0.90       169
    Leak_L_M       1.00      0.75      0.86       187
    Leak_L_N       1.00      0.76      0.86       192
    Leak_N_O       1.00      0.72    

In [19]:
import sklearn; 
print(sklearn.__version__)

1.7.2


In [20]:
import pandas; 
print(pandas.__version__)

2.3.3
